In [1]:
import os
import glob
import numpy as np
import pandas as pd
import gc
import time

PATH_DATASET = "/kaggle/input/icecube-neutrinos-in-deep-ice"
DATA_DIR = PATH_DATASET

os.environ.setdefault("PYTHONHASHSEED", "0")

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

np.seterr(all="ignore")



{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

In [2]:
# Speed: build a sensor_id -> (x,y,z) LUT once; keep float32 and avoid per-batch joins.
geometry = pd.read_csv(os.path.join(DATA_DIR, "sensor_geometry.csv"))
geometry.set_index("sensor_id", inplace=True)
geometry = geometry.apply(np.float32)

_geom_sensor_ids = geometry.index.to_numpy()
_geom_xyz = geometry[["x", "y", "z"]].to_numpy(dtype=np.float32, copy=False)

max_sid = int(_geom_sensor_ids.max())
_geom_xyz_lut = np.full((max_sid + 1, 3), np.nan, dtype=np.float32)
_geom_xyz_lut[_geom_sensor_ids.astype(np.int64, copy=False)] = _geom_xyz

del geometry
gc.collect()



0

In [3]:
import math
import warnings

warnings.filterwarnings("ignore")


def cartesian_to_sphere(x, y, z):
    x = float(x)
    y = float(y)
    z = float(z)
    x2y2 = x * x + y * y
    r2 = x2y2 + z * z
    r = math.sqrt(r2) if r2 > 0 else 0.0

    if r == 0.0:
        zenith = 0.0
    else:
        cz = max(-1.0, min(1.0, z / r))
        zenith = math.acos(cz)

    azimuth = math.atan2(y, x)
    if azimuth < 0:
        azimuth += 2 * math.pi
    return azimuth, zenith


def adjust_sphere(azimuth, zenith):
    azimuth = float(azimuth)
    zenith = float(zenith)
    if zenith < 0:
        zenith += math.pi
        azimuth += math.pi
    if azimuth < 0:
        azimuth += math.pi * 2
    azimuth = azimuth % (2 * math.pi)
    if zenith < 0:
        zenith = 0.0
    if zenith > math.pi:
        zenith = math.pi
    return azimuth, zenith


def sphere_to_cartesian(azimuth, zenith):
    azimuth = float(azimuth)
    zenith = float(zenith)
    x = math.cos(azimuth) * math.sin(zenith)
    y = math.sin(azimuth) * math.sin(zenith)
    z = math.cos(zenith)
    return x, y, z




In [4]:
def compute_direction(r, t):
    """compute Line-fit using r vector and time vectors
    return v_est and r_est"""

    def avg(p):
        if len(p.shape) == 1:
            p = p.reshape(-1, 1)
        return np.mean(p, axis=0)

    q = avg(t**2) - avg(t) ** 2
    if np.allclose(q, 0):
        v_est = np.zeros((3,), dtype=np.float32)
        r_est = avg(r).astype(np.float32)
        return v_est, r_est

    v_est = (avg(r * t) - avg(r) * avg(t)) / q
    r_est = avg(r) - v_est * avg(t)
    return v_est, r_est




In [5]:
sample_csv_path = os.path.join(PATH_DATASET, "sample_submission.csv")
if not os.path.exists(sample_csv_path):
    sample_csv_path = "/kaggle/input/sample_submission.csv"

sample_parquet_path = os.path.join(PATH_DATASET, "sample_submission.parquet")
if not os.path.exists(sample_parquet_path):
    sample_parquet_path = os.path.join(
        PATH_DATASET, "icecube-neutrinos-in-deep-ice", "sample_submission.parquet"
    )
if not os.path.exists(sample_parquet_path):
    sample_parquet_path = None

test_meta_path = os.path.join(PATH_DATASET, "test_meta.parquet")
if not os.path.exists(test_meta_path):
    test_meta_path = os.path.join(
        PATH_DATASET, "icecube-neutrinos-in-deep-ice", "test_meta.parquet"
    )

test_meta = pd.read_parquet(test_meta_path, columns=["event_id"])
_event_ids_all = test_meta["event_id"].to_numpy(copy=False).astype(np.int64, copy=False)
del test_meta
gc.collect()

ls = glob.glob(os.path.join(PATH_DATASET, "test", "*.parquet"))
if len(ls) == 0:
    ls = glob.glob(
        os.path.join(PATH_DATASET, "icecube-neutrinos-in-deep-ice", "test", "*.parquet")
    )
ls = sorted(ls)
print(f"Found {len(ls)} test batch parquet files")

TOPK = 800

_TWO_PI = np.float32(2.0 * np.pi)
_PI = np.float32(np.pi)


def _cart_to_sphere_adjust_np(vx, vy, vz):
    vx = vx.astype(np.float32, copy=False)
    vy = vy.astype(np.float32, copy=False)
    vz = vz.astype(np.float32, copy=False)

    r = np.sqrt(vx * vx + vy * vy + vz * vz, dtype=np.float32)
    zenith = np.zeros_like(r, dtype=np.float32)
    nz = r > 0
    cz = np.empty_like(r, dtype=np.float32)
    cz[nz] = vz[nz] / r[nz]
    np.clip(cz, -1.0, 1.0, out=cz)
    zenith[nz] = np.arccos(cz[nz]).astype(np.float32, copy=False)

    azimuth = np.arctan2(vy, vx).astype(np.float32, copy=False)
    azimuth = np.where(azimuth < 0, azimuth + _TWO_PI, azimuth).astype(
        np.float32, copy=False
    )

    negz = zenith < 0
    if np.any(negz):
        zenith = np.where(negz, zenith + _PI, zenith).astype(np.float32, copy=False)
        azimuth = np.where(negz, azimuth + _PI, azimuth).astype(np.float32, copy=False)

    negaz = azimuth < 0
    if np.any(negaz):
        azimuth = np.where(negaz, azimuth + _TWO_PI, azimuth).astype(
            np.float32, copy=False
        )

    azimuth = np.mod(azimuth, _TWO_PI).astype(np.float32, copy=False)
    zenith = np.clip(zenith, 0.0, _PI).astype(np.float32, copy=False)
    return azimuth, zenith




Found 66 test batch parquet files


In [6]:
# Speed: replace sorting-heavy TOPK selection with per-event argpartition + stable tie-break by original order.
# This preserves semantics: select up to TOPK highest-charge pulses per event from base_mask, stable within equal charge.
def _topk_per_group_stable(inv, charge, k):
    """
    inv: int32 group id per row, assumed 0..n_groups-1 (not necessarily contiguous in data)
    charge: float32 per row
    k: int TOPK
    returns a boolean keep mask of same length as inv
    """
    n = inv.size
    if n == 0:
        return np.zeros(0, dtype=bool)

    # Group rows by inv using a stable sort, so later we can break ties by original order (stable).
    order = np.argsort(inv, kind="stable")
    inv_s = inv[order]
    ch_s = charge[order]

    # Group boundaries
    newgrp = np.empty(n, dtype=bool)
    newgrp[0] = True
    newgrp[1:] = inv_s[1:] != inv_s[:-1]
    starts = np.flatnonzero(newgrp)
    ends = np.empty_like(starts)
    ends[:-1] = starts[1:]
    ends[-1] = n

    keep_s = np.zeros(n, dtype=bool)

    # Per-group selection: argpartition to get top-k by charge, then stable sort those k by (-charge, order_index).
    # This matches the previous lexsort((-charge, group)) behavior, which effectively keeps earlier rows on ties.
    for s, e in zip(starts, ends):
        m = e - s
        if m <= k:
            keep_s[s:e] = True
            continue

        seg = ch_s[s:e]
        # indices of k largest (unordered)
        idx_part = np.argpartition(seg, -k)[-k:]
        # stable tie-break: by -charge then by original within-group order (which is stable due to argsort(inv, stable))
        # idx_part is within [0, m); add s to refer to keep_s slice
        idx_sorted = idx_part[
            np.lexsort((idx_part.astype(np.int32, copy=False), -seg[idx_part]))
        ]
        keep_s[s + idx_sorted] = True

    keep = np.zeros(n, dtype=bool)
    keep[order] = keep_s
    return keep


_gc_every = 24  # fewer full GC cycles reduces overhead substantially
_pred_event_list = []
_pred_az_list = []
_pred_ze_list = []

for bi, batch_file in enumerate(ls):
    t0b = time.time()
    base = os.path.basename(batch_file)
    try:
        batch_id = int(base.replace("batch_", "").replace(".parquet", ""))
    except Exception:
        batch_id = None

    print(f"processing: {batch_file}")

    df = pd.read_parquet(
        batch_file,
        columns=["time", "sensor_id", "charge", "auxiliary"],
        engine="pyarrow",
    )

    if df.index.name == "event_id":
        event = df.index.to_numpy(copy=False)
    elif "event_id" in df.columns:
        event = df["event_id"].to_numpy(copy=False)
    else:
        df = df.reset_index().rename(columns={"index": "event_id"})
        event = df["event_id"].to_numpy(copy=False)

    # Speed: convert/cast once and reuse; avoid repeated astype in hot path.
    sensor = df["sensor_id"].to_numpy(copy=False).astype(np.int64, copy=False)
    df_time = df["time"].to_numpy(copy=False).astype(np.float32, copy=False)
    df_aux = df["auxiliary"].to_numpy(copy=False)
    df_charge = df["charge"].to_numpy(copy=False).astype(np.float32, copy=False)
    event = event.astype(np.int64, copy=False)

    xyz_all = _geom_xyz_lut[sensor]
    geom_valid = ~np.isnan(xyz_all[:, 0])
    base_mask = (~df_aux) & geom_valid  # time NaN check handled by np.isfinite below
    if np.any(base_mask):
        base_mask &= np.isfinite(df_time)

    if not np.any(base_mask):
        del (
            df,
            event,
            sensor,
            df_time,
            df_aux,
            df_charge,
            xyz_all,
            geom_valid,
            base_mask,
        )
        if (bi + 1) % _gc_every == 0:
            gc.collect()
        print(f"done batch {batch_id} in {time.time() - t0b:.2f}s (no valid pulses)")
        continue

    idx0 = np.flatnonzero(base_mask)
    ev0 = event[idx0]

    uniq_ev, inv0 = np.unique(ev0, return_inverse=True)
    inv0 = inv0.astype(np.int32, copy=False)
    n_events = int(uniq_ev.size)

    if TOPK is not None and TOPK > 0:
        ch0 = df_charge[idx0]
        keep0 = _topk_per_group_stable(inv0, ch0, int(TOPK))
        idx_sel = idx0[keep0]
        inv_sel = inv0[keep0]
        del ch0, keep0
    else:
        idx_sel = idx0
        inv_sel = inv0

    if idx_sel.size == 0:
        del (
            df,
            event,
            sensor,
            df_time,
            df_aux,
            df_charge,
            xyz_all,
            geom_valid,
            base_mask,
            idx0,
            ev0,
            inv0,
            uniq_ev,
            idx_sel,
            inv_sel,
        )
        if (bi + 1) % _gc_every == 0:
            gc.collect()
        print(
            f"done batch {batch_id} in {time.time() - t0b:.2f}s (no pulses after TOPK)"
        )
        continue

    t_sel = df_time[idx_sel]
    xyz_sel = xyz_all[idx_sel]

    inv_sel64 = inv_sel.astype(np.int64, copy=False)

    cnt = np.bincount(inv_sel64, minlength=n_events).astype(np.float32, copy=False)
    valid_ev_mask = cnt > 0
    if not np.any(valid_ev_mask):
        del (
            df,
            event,
            sensor,
            df_time,
            df_aux,
            df_charge,
            xyz_all,
            geom_valid,
            base_mask,
            idx0,
            ev0,
            inv0,
            uniq_ev,
            idx_sel,
            inv_sel,
            inv_sel64,
            t_sel,
            xyz_sel,
            cnt,
            valid_ev_mask,
        )
        if (bi + 1) % _gc_every == 0:
            gc.collect()
        print(
            f"done batch {batch_id} in {time.time() - t0b:.2f}s (no events after TOPK)"
        )
        continue

    sum_t = np.bincount(inv_sel64, weights=t_sel, minlength=n_events).astype(
        np.float32, copy=False
    )
    t2 = t_sel * t_sel
    sum_t2 = np.bincount(inv_sel64, weights=t2, minlength=n_events).astype(
        np.float32, copy=False
    )

    x = xyz_sel[:, 0].astype(np.float32, copy=False)
    y = xyz_sel[:, 1].astype(np.float32, copy=False)
    z = xyz_sel[:, 2].astype(np.float32, copy=False)

    sum_x = np.bincount(inv_sel64, weights=x, minlength=n_events).astype(
        np.float32, copy=False
    )
    sum_y = np.bincount(inv_sel64, weights=y, minlength=n_events).astype(
        np.float32, copy=False
    )
    sum_z = np.bincount(inv_sel64, weights=z, minlength=n_events).astype(
        np.float32, copy=False
    )

    xt = x * t_sel
    yt = y * t_sel
    zt = z * t_sel
    sum_xt = np.bincount(inv_sel64, weights=xt, minlength=n_events).astype(
        np.float32, copy=False
    )
    sum_yt = np.bincount(inv_sel64, weights=yt, minlength=n_events).astype(
        np.float32, copy=False
    )
    sum_zt = np.bincount(inv_sel64, weights=zt, minlength=n_events).astype(
        np.float32, copy=False
    )

    inv_cnt = np.zeros_like(cnt, dtype=np.float32)
    inv_cnt[valid_ev_mask] = 1.0 / cnt[valid_ev_mask]

    mean_t = sum_t * inv_cnt
    mean_t2 = sum_t2 * inv_cnt

    mean_x = sum_x * inv_cnt
    mean_y = sum_y * inv_cnt
    mean_z = sum_z * inv_cnt

    mean_xt = sum_xt * inv_cnt
    mean_yt = sum_yt * inv_cnt
    mean_zt = sum_zt * inv_cnt

    q = mean_t2 - mean_t * mean_t
    q_zero = np.abs(q) <= 1e-8
    denom = np.where(q_zero, 1.0, q).astype(np.float32, copy=False)

    vx = (mean_xt - mean_x * mean_t) / denom
    vy = (mean_yt - mean_y * mean_t) / denom
    vz = (mean_zt - mean_z * mean_t) / denom

    vx[q_zero] = 0.0
    vy[q_zero] = 0.0
    vz[q_zero] = 0.0

    vx = -vx
    vy = -vy
    vz = -vz

    valid_idx = np.flatnonzero(valid_ev_mask)
    az_v, ze_v = _cart_to_sphere_adjust_np(vx[valid_idx], vy[valid_idx], vz[valid_idx])
    ev_out = uniq_ev[valid_idx].astype(np.int64, copy=False)

    _pred_event_list.append(ev_out)
    _pred_az_list.append(az_v.astype(np.float32, copy=False))
    _pred_ze_list.append(ze_v.astype(np.float32, copy=False))

    del (
        df,
        event,
        sensor,
        df_time,
        df_aux,
        df_charge,
        xyz_all,
        geom_valid,
        base_mask,
        idx0,
        ev0,
        inv0,
        idx_sel,
        inv_sel,
        inv_sel64,
        t_sel,
        xyz_sel,
        cnt,
        valid_ev_mask,
        sum_t,
        sum_t2,
        t2,
        sum_x,
        sum_y,
        sum_z,
        xt,
        yt,
        zt,
        sum_xt,
        sum_yt,
        sum_zt,
        inv_cnt,
        mean_t,
        mean_t2,
        mean_x,
        mean_y,
        mean_z,
        mean_xt,
        mean_yt,
        mean_zt,
        q,
        q_zero,
        denom,
        vx,
        vy,
        vz,
        x,
        y,
        z,
        valid_idx,
        az_v,
        ze_v,
        ev_out,
        uniq_ev,
    )
    if (bi + 1) % _gc_every == 0:
        gc.collect()
    print(f"done batch {batch_id} in {time.time() - t0b:.2f}s")



processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_104.parquet
done batch 104 in 9.90s
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_128.parquet
done batch 128 in 9.39s
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_141.parquet
done batch 141 in 8.02s
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_143.parquet
done batch 143 in 7.95s
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_15.parquet
done batch 15 in 8.08s
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_160.parquet
done batch 160 in 8.04s
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_163.parquet
done batch 163 in 8.47s
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_171.parquet
done batch 171 in 9.02s
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_176.parquet
done batch 176 in 8.12s
processing: /kaggle/input/icecube-neutrinos-in-deep-ice/test/batch_201.parq

In [7]:
event_ids = _event_ids_all  # already int64
n = event_ids.shape[0]

az = np.zeros(n, dtype=np.float32)
ze = np.zeros(n, dtype=np.float32)

if len(_pred_event_list):
    pred_keys = np.concatenate(_pred_event_list).astype(np.int64, copy=False)
    pred_az = np.concatenate(_pred_az_list).astype(np.float32, copy=False)
    pred_ze = np.concatenate(_pred_ze_list).astype(np.float32, copy=False)
else:
    pred_keys = np.empty(0, dtype=np.int64)
    pred_az = np.empty(0, dtype=np.float32)
    pred_ze = np.empty(0, dtype=np.float32)

_pred_event_list = None
_pred_az_list = None
_pred_ze_list = None
gc.collect()

if pred_keys.size:
    sort_idx = np.argsort(pred_keys)
    k_sorted = pred_keys[sort_idx]
    az_sorted = pred_az[sort_idx]
    ze_sorted = pred_ze[sort_idx]

    pos = np.searchsorted(k_sorted, event_ids)
    in_range = pos < k_sorted.size
    match = np.zeros_like(in_range, dtype=bool)
    match[in_range] = k_sorted[pos[in_range]] == event_ids[in_range]

    az[match] = az_sorted[pos[match]]
    ze[match] = ze_sorted[pos[match]]

out_path = "submission.csv"

with open(out_path, "w") as f:
    f.write("event_id,azimuth,zenith\n")
    chunk = 1_000_000
    for i in range(0, n, chunk):
        j = min(i + chunk, n)
        block = np.column_stack((event_ids[i:j], az[i:j], ze[i:j]))
        np.savetxt(
            f,
            block,
            fmt=["%d", "%.9g", "%.9g"],
            delimiter=",",
        )

print("Wrote submission.csv with rows:", n)

with open("submission.csv", "r") as f:
    for _ in range(5):
        print(f.readline().rstrip("\n"))

Wrote submission.csv with rows: 13200000
event_id,azimuth,zenith
45566128,4.90351391,1.48593104
45566141,4.32831669,2.11865616
45566144,2.45727563,0.362875104
45566153,5.56009912,1.55215788
